# 📚 Capítulo 4: Recursión — Ejemplos Ilustrativos

## Estructuras de Datos y Algoritmos - Lic. en Sistemas

## 📋 Información del Material
- **Capítulo**: 4 (Notebook 2/3)
- **Título**: Ejemplos Ilustrativos de Recursión
- **Fuente**: Goodrich, Tamassia & Goldwasser — Secciones 4.1 y 4.4
- **Tipo**: Material Teórico
- **Fecha**: 2026-03-05

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap04/02_Ejemplos_Recursion_Teoria.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, el estudiante será capaz de:

1. **Implementar búsqueda binaria de forma recursiva** y trazar su ejecución
2. **Modelar la exploración del sistema de archivos** como problema recursivo
3. **Resolver las Torres de Hanói** y entender por qué cuesta O(2^n)
4. **Comparar Fibonacci ingenuo vs memoizado** y medir la diferencia en tiempo

---


## 📖 Contenidos

### 🔍 Sección 1 — Búsqueda Binaria Recursiva
1. Revisión de la búsqueda binaria iterativa
2. Versión recursiva con índices lo/hi
3. Traza y verificación con asserts

### 📂 Sección 2 — Sistema de Archivos (Disk Usage)
1. Analogía: carpetas y archivos como árbol
2. Función recursiva `disk_usage`
3. Impresión del árbol de directorios

### 🗼 Sección 3 — Torres de Hanói
1. Descripción del problema
2. Solución recursiva elegante: O(2^n)
3. Traza de movimientos paso a paso

### 🐇 Sección 4 — Fibonacci: Peligros y Soluciones
1. Fibonacci recursivo ingenuo: O(2^n)
2. Por qué es tan lento (árbol de llamadas)
3. Memoización con `@functools.lru_cache`

---


## 🔧 Configuración del Entorno

Importamos las librerías necesarias para este capítulo:

In [ ]:
# Librerías estándar de Python
import sys
import time
import math
from typing import List, Optional, Any, Union

# Librerías para visualización y análisis
import matplotlib.pyplot as plt
import numpy as np

# Configuración de matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"Python version: {sys.version}")
print(f"Entorno configurado correctamente ✅")

---

# 🔍 Sección 1: Búsqueda Binaria Recursiva

## Concepto

La búsqueda binaria resuelve el siguiente problema:
> Dado un arreglo **ordenado** `data` y un valor `target`, devolver el índice de `target` o `−1` si no existe.

**Idea recursiva**: Compara `target` con el elemento del centro.

- Si igual → encontrado.
- Si menor → buscar en la mitad izquierda.
- Si mayor → buscar en la mitad derecha.

```
[2, 4, 7, 10, 15, 21, 33, 45]   buscar 15
              ↑ centro (índice 3) = 10  →  15 > 10, buscar derecha
                       [15, 21, 33, 45]
                        ↑ centro = 21  →  15 < 21, buscar izquierda
                        [15]
                         ↑ encontrado en índice 4   ✅
```

**T(n) = T(n/2) + O(1)  →  O(log n)**


In [ ]:
# ─────────────────────────────────────────────────────────────
# Búsqueda binaria recursiva — Goodrich 4.1
# ─────────────────────────────────────────────────────────────
from typing import List

def binary_search(data: List[int], target: int, lo: int, hi: int) -> int:
    """
    Busca target en data[lo..hi] (inclusive).
    Retorna el índice si lo encuentra, -1 si no.

    Caso base:  lo > hi  →  -1  (rango vacío)
    Caso recursivo: comparar con mid y recursar en subarreglo
    Complejidad: O(log n) tiempo, O(log n) espacio (frames)
    """
    if lo > hi:
        return -1               # caso base: no encontrado
    mid = (lo + hi) // 2
    if data[mid] == target:
        return mid              # caso base: encontrado
    elif target < data[mid]:
        return binary_search(data, target, lo, mid - 1)   # mitad izquierda
    else:
        return binary_search(data, target, mid + 1, hi)   # mitad derecha

# Interfaz más cómoda
def buscar(data: List[int], target: int) -> int:
    return binary_search(data, target, 0, len(data) - 1)

# Demo
datos = [2, 4, 7, 10, 15, 21, 33, 45, 57, 68, 79, 85, 93]
print("Arreglo:", datos)
print()
for t in [15, 2, 93, 50, 4]:
    idx = buscar(datos, t)
    encontrado = f"índice {idx}" if idx != -1 else "no encontrado"
    print(f"  buscar({t:>3}) → {encontrado}")

# Validación
assert buscar(datos, 15) == 4
assert buscar(datos, 2)  == 0
assert buscar(datos, 93) == len(datos) - 1
assert buscar(datos, 50) == -1
print("\nTodos los asserts pasaron ✅")


### 🔍 Por qué funciona la búsqueda binaria recursiva

Cada llamada descarta la mitad del arreglo que queda:
- Llamada 1: busca en n elementos
- Llamada 2: busca en n/2 elementos
- Llamada 3: busca en n/4 elementos
- ...
- Llamada k: busca en n/2^k elementos → termina cuando n/2^k ≤ 1, es decir **k ≤ log₂(n)**

**Complejidad:**
- **T(n)**: O(log n) — la recurrencia T(n) = T(n/2) + 1 resuelve a O(log n)
- **S(n)**: O(log n) — log n frames en la pila simultáneamente

---


---

# 📂 Sección 2: Sistema de Archivos (Disk Usage)

## Concepto

Un sistema de archivos tiene estructura de **árbol**: las carpetas contienen sub-carpetas y archivos.

```
/home/
├── user/               (128 KB propia + suma de hijos)
│   ├── docs/           (64 KB + archivos internos)
│   │   ├── tesis.pdf   (1024 KB)
│   │   └── notas.txt   (5 KB)
│   └── img/            (0 KB)
│       └── foto.png    (256 KB)
└── .bashrc             (4 KB)
```

**Idea recursiva**:
> `disk_usage(path)` = tamaño de `path` + suma(`disk_usage(hijo)` para cada hijo)

**La recursión funciona perfectamente aquí porque el sistema de archivos ES un árbol.**


In [ ]:
# ─────────────────────────────────────────────────────────────
# Disk Usage sobre un árbol de nodos simulados
# (sin acceso real al FS; usando clases para simular)
# ─────────────────────────────────────────────────────────────
from __future__ import annotations
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Nodo:
    """Simula una entrada del sistema de archivos."""
    nombre: str
    tamanio: int                          # KB propios
    hijos: List['Nodo'] = field(default_factory=list)

    def es_hoja(self) -> bool:
        return len(self.hijos) == 0

def disk_usage(nodo: Nodo, nivel: int = 0) -> int:
    """
    Retorna el uso total de espacio en KB del subárbol rooted en nodo.
    Complejidad: O(n) donde n = número de nodos del árbol.
    """
    total = nodo.tamanio
    indent = "  " * nivel
    for hijo in nodo.hijos:
        total += disk_usage(hijo, nivel + 1)
    print(f"{indent}{'📄' if nodo.es_hoja() else '📁'} {nodo.nombre:20s} {total:>8} KB")
    return total

# Construcción del árbol simulado
home = Nodo("home/", 0, [
    Nodo("user/", 128, [
        Nodo("docs/", 64, [
            Nodo("tesis.pdf", 1024),
            Nodo("notas.txt", 5),
        ]),
        Nodo("img/", 0, [
            Nodo("foto.png", 256),
        ]),
    ]),
    Nodo(".bashrc", 4),
])

print("Uso de disco por carpeta (KB):")
print("─" * 45)
total = disk_usage(home)
print("─" * 45)
print(f"Total /home: {total} KB")


---

# 🗼 Sección 3: Torres de Hanói

## El problema

Tienes **n discos** de distinto tamaño sobre 3 clavijas (A, B, C).  
Los discos están apilados de mayor a menor en la clavija A.

**Objetivo**: Mover todos los discos a la clavija C siguiendo estas reglas:
1. Solo puedes mover **un disco** a la vez
2. No puedes colocar un disco más grande sobre uno más pequeño

**¿Cuántos movimientos mínimos se necesitan?** Spoiler: **2^n − 1**

### Solución recursiva

```
hanoi(n, origen, destino, auxiliar):
    caso base: n == 0 → no hacer nada
    1. Mover n-1 discos de origen → auxiliar  (usando destino como aux)
    2. Mover disco n de origen → destino       (el movimiento directo)
    3. Mover n-1 discos de auxiliar → destino (usando origen como aux)
```

**T(n) = 2·T(n-1) + 1  →  O(2^n)**  Esta recursión no tiene "truco" para mejorarla; es inherentemente exponencial.


In [ ]:
def hanoi(n: int, origen: str, destino: str, auxiliar: str) -> int:
    """
    Resuelve Torres de Hanói moviendo n discos de 'origen' a 'destino'.
    Retorna el número de movimientos realizados.
    Complejidad: O(2^n)
    """
    if n == 0:
        return 0
    # Paso 1: mover n-1 discos al auxiliar
    movs = hanoi(n - 1, origen, auxiliar, destino)
    # Paso 2: mover el disco más grande al destino
    print(f"  Mover disco {n} de {origen} → {destino}")
    movs += 1
    # Paso 3: mover n-1 discos del auxiliar al destino
    movs += hanoi(n - 1, auxiliar, destino, origen)
    return movs


print("=" * 42)
print("Torres de Hanói con n=3")
print("=" * 42)
total = hanoi(3, "A", "C", "B")
print(f"\nTotal de movimientos: {total}  (esperado: 2^3 - 1 = 7)")

# Verificar la fórmula 2^n - 1 para varios n
for n in range(0, 7):
    movs = hanoi(n, "A", "C", "B")
    esperado = 2**n - 1
    assert movs == esperado, f"n={n}: got {movs}, expected {esperado}"
print("\n✓ Todos los casos verificados: movimientos == 2^n - 1")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualizar el crecimiento exponencial de movimientos
ns = list(range(0, 21))
movimientos = [2**n - 1 for n in ns]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Escala lineal — crecimiento explosivo
axes[0].plot(ns, movimientos, 'o-', color='crimson', linewidth=2, markersize=5)
axes[0].set_title("Movimientos Hanói (escala lineal)", fontsize=13)
axes[0].set_xlabel("n (número de discos)")
axes[0].set_ylabel("Movimientos (2ⁿ − 1)")
axes[0].grid(True, alpha=0.4)
axes[0].fill_between(ns, movimientos, alpha=0.15, color='crimson')

# Escala log — confirma exponencial → línea recta
axes[1].semilogy(ns[1:], movimientos[1:], 's-', color='darkorange', linewidth=2, markersize=5)
axes[1].set_title("Crecimiento O(2ⁿ) (escala logarítmica)", fontsize=13)
axes[1].set_xlabel("n (número de discos)")
axes[1].set_ylabel("log(Movimientos)")
axes[1].grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.suptitle("Torres de Hanói: crecimiento exponencial", fontsize=14, y=1.02)
plt.show()

print(f"Para n=30 discos necesitaríamos {2**30 - 1:,} movimientos")
print(f"Si cada movimiento toma 1 segundo → {(2**30 - 1) / 3600 / 24 / 365:.1f} años")


---

# 🐇 Sección 4: Fibonacci — de O(2^n) a O(n) con Memoización

## Sucesión de Fibonacci

$$F(0) = 0,\quad F(1) = 1,\quad F(n) = F(n-1) + F(n-2)$$

La implementación recursiva directa parece natural pero **recalcula los mismos valores exponencialmente** muchas veces.

### Árbol de llamadas para fib(5)

```
fib(5)
├── fib(4)
│   ├── fib(3)
│   │   ├── fib(2)
│   │   │   ├── fib(1) = 1
│   │   │   └── fib(0) = 0
│   │   └── fib(1) = 1
│   └── fib(2)          ← ¡calculado de nuevo!
│       ├── fib(1) = 1
│       └── fib(0) = 0
└── fib(3)              ← ¡calculado de nuevo!
    ├── fib(2)          ← ¡y otra vez!
    │   ├── fib(1) = 1
    │   └── fib(0) = 0
    └── fib(1) = 1
```

**Memoización**: guardar los resultados ya calculados. `@functools.lru_cache` hace esto automáticamente.

| Versión | Complejidad temporal | Complejidad espacial |
|---------|---------------------|---------------------|
| Naïve   | O(2^n)              | O(n) pila           |
| Memo    | O(n)                | O(n) caché          |
| Bottom-up (iterativa) | O(n) | O(1) |


In [ ]:
import functools
import sys

# Aumentar límite para la versión naïve en pruebas pequeñas
sys.setrecursionlimit(5000)

# --- Versión naïve O(2^n) ---
llamadas_naive = 0

def fib_naive(n: int) -> int:
    """Fibonacci recursivo sin memoización. O(2^n)."""
    global llamadas_naive
    llamadas_naive += 1
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


# --- Versión con memoización O(n) ---
llamadas_memo = 0

@functools.lru_cache(maxsize=None)
def fib_memo(n: int) -> int:
    """Fibonacci con memoización automática. O(n)."""
    global llamadas_memo
    llamadas_memo += 1
    if n <= 1:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)


# --- Versión bottom-up iterativa O(n) tiempo, O(1) espacio ---
def fib_iterativo(n: int) -> int:
    """Fibonacci iterativo bottom-up. O(n) tiempo, O(1) espacio."""
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


# Verificar correctitud
for i in range(10):
    v1 = fib_naive(i)
    fib_memo.cache_clear()
    v2 = fib_memo(i)
    v3 = fib_iterativo(i)
    assert v1 == v2 == v3, f"Discrepancia en fib({i}): {v1}, {v2}, {v3}"

print("✓ Las tres versiones producen resultados idénticos")
print(f"  fib(9) = {fib_iterativo(9)}")
print(f"  fib(20) = {fib_iterativo(20)}")
print(f"  fib(50) = {fib_iterativo(50)}")


In [ ]:
import time
import matplotlib.pyplot as plt

# Comparación de llamadas recursivas: naïve vs memo
ns_small = list(range(0, 26))
counts_naive = []
counts_memo  = []

for n in ns_small:
    # Naïve: contar llamadas
    global llamadas_naive, llamadas_memo
    llamadas_naive = 0
    fib_naive(n)
    counts_naive.append(llamadas_naive)

    # Memo: contar llamadas (cache limpiado entre iteraciones, excepto para
    # mostrar que memo también se beneficia del cache acumulado)
    fib_memo.cache_clear()
    llamadas_memo = 0
    fib_memo(n)
    counts_memo.append(llamadas_memo)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ns_small, counts_naive, 'o-', color='crimson',  label='Naïve O(2ⁿ)', linewidth=2)
axes[0].plot(ns_small, counts_memo,  's-', color='steelblue', label='Memo  O(n)',    linewidth=2)
axes[0].set_title("Llamadas a fib(n): Naïve vs Memo", fontsize=13)
axes[0].set_xlabel("n")
axes[0].set_ylabel("Número de llamadas recursivas")
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Escala logarítmica para visualizar mejor
axes[1].semilogy(ns_small[1:], counts_naive[1:], 'o-', color='crimson',  label='Naïve O(2ⁿ)', linewidth=2)
axes[1].semilogy(ns_small[1:], counts_memo[1:],  's-', color='steelblue', label='Memo  O(n)',   linewidth=2)
axes[1].set_title("Mismo gráfico en escala logarítmica", fontsize=13)
axes[1].set_xlabel("n")
axes[1].set_ylabel("log(llamadas)")
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.show()

print(f"\nPara n=25:")
print(f"  Naïve hizo {counts_naive[25]:,} llamadas")
print(f"  Memo  hizo {counts_memo[25]:,} llamadas  ({counts_naive[25]//max(counts_memo[25],1)}× menos)")


In [ ]:
import unittest

class TestEjemplosRecursion(unittest.TestCase):

    # ---- Búsqueda Binaria ----
    def test_binary_search_encontrado(self):
        data = [2, 5, 8, 12, 16, 23, 38, 56]
        self.assertEqual(binary_search(data, 23, 0, len(data)-1), 5)

    def test_binary_search_primer_elemento(self):
        data = list(range(0, 20, 2))
        self.assertEqual(binary_search(data, 0, 0, len(data)-1), 0)

    def test_binary_search_no_encontrado(self):
        data = [1, 3, 5, 7, 9]
        self.assertEqual(binary_search(data, 4, 0, len(data)-1), -1)

    # ---- Fibonacci ----
    def test_fibonacci_valores_conocidos(self):
        esperados = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
        for i, v in enumerate(esperados):
            self.assertEqual(fib_iterativo(i), v, f"fib({i}) debería ser {v}")

    def test_fibonacci_memo_igual_iterativo(self):
        fib_memo.cache_clear()
        for n in range(30):
            self.assertEqual(fib_memo(n), fib_iterativo(n))

    # ---- Torres de Hanói ----
    def test_hanoi_formula(self):
        import io, contextlib
        for n in range(1, 8):
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                movs = hanoi(n, "A", "C", "B")
            self.assertEqual(movs, 2**n - 1, f"Hanói({n}) esperaba {2**n-1} movimientos")


loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestEjemplosRecursion)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print("\n✓ Suite completa –", result.testsRun, "tests,", len(result.failures), "fallos")


In [ ]:
import time

# Comparación de rendimiento naïve vs memo vs iterativo para Fibonacci
print(f"{'n':>5} | {'Naive (s)':>12} | {'Memo (s)':>12} | {'Iter (s)':>12}")
print("-" * 52)

for n in [10, 15, 20, 25, 30]:
    # Naïve
    t0 = time.perf_counter()
    fib_naive(n)
    t_naive = time.perf_counter() - t0

    # Memo (precalentar caché)
    fib_memo.cache_clear()
    t0 = time.perf_counter()
    fib_memo(n)
    t_memo = time.perf_counter() - t0

    # Iterativo
    t0 = time.perf_counter()
    fib_iterativo(n)
    t_iter = time.perf_counter() - t0

    print(f"{n:>5} | {t_naive:>12.6f} | {t_memo:>12.6f} | {t_iter:>12.6f}")

print()
print("Nota: para n > 35, el naïve se vuelve imprácticamente lento.")
print("La memoización lo convierte de prohibitivo a instantáneo.")


In [ ]:
# Demostración de disk_usage con un árbol de directorios realista
from typing import List
from dataclasses import dataclass, field

@dataclass
class Nodo:
    nombre: str
    tamanio: int
    hijos: List['Nodo'] = field(default_factory=list)
    def es_hoja(self) -> bool: return len(self.hijos) == 0

def disk_usage(nodo: Nodo, nivel: int = 0) -> int:
    total = nodo.tamanio
    for hijo in nodo.hijos:
        total += disk_usage(hijo, nivel + 1)
    icono = "📄" if nodo.es_hoja() else "📁"
    print(f"{'  ' * nivel}{icono} {nodo.nombre:25s} {total:>8} KB")
    return total

# Árbol de ejemplo
raiz = Nodo("proyecto/", 0, [
    Nodo("src/", 0, [
        Nodo("main.py",    12),
        Nodo("utils.py",   8),
        Nodo("models/", 0, [
            Nodo("user.py",   5),
            Nodo("item.py",   7),
        ]),
    ]),
    Nodo("tests/", 0, [
        Nodo("test_main.py", 3),
        Nodo("test_utils.py", 4),
    ]),
    Nodo("README.md",   2),
    Nodo("requirements.txt", 1),
])

print("Árbol de archivos con tamaños acumulados:")
print("=" * 45)
total = disk_usage(raiz)
print("=" * 45)
print(f"Total del proyecto: {total} KB")


---

## 📊 Resumen: Los Cuatro Ejemplos Clásicos

### Tabla comparativa de complejidad

| Algoritmo | Estrategia recursiva | Tiempo | Espacio | Clave del diseño |
|-----------|----------------------|--------|---------|-----------------|
| Búsqueda Binaria | Divide: descarta mitad | O(log n) | O(log n) pila | rango `[lo, hi]` en parámetros |
| Uso de Disco | DFS postorden en árbol | O(n) | O(h) profundidad | acumular en retorno |
| Torres de Hanói | Divide en 3 subproblemas | **O(2^n)** | O(n) pila | exponencial inevitable |
| Fibonacci (naïve) | Dos llamadas recursivas | **O(2^n)** | O(n) pila | ¡subproblemas solapados! |
| Fibonacci (memo) | + caché de resultados | O(n) | O(n) caché | memoización resuelve el solapamiento |

### 🔑 Lecciones transversales

1. **Divide y vencerás** reduce complejidad cuando los subproblemas son *disjuntos* (binary search, disk usage)
2. **Subproblemas solapados** → memoización o programación dinámica (Fibonacci)
3. **Problemas "exponencialmente difíciles"** como Hanói no pueden mejorarse: el output mismo es de tamaño O(2^n)
4. **El espacio importa**: cada llamada recursiva añade un marco a la pila — la profundidad determina el espacio

### 🗺️ Próximo notebook

En el **Notebook 3** analizaremos los *tipos* de recursión (lineal, binaria, múltiple, de cola), las relaciones de recurrencia T(n) y técnicas de diseño recursivo top-down.


---

## ✏️ Autoevaluación

Reflexiona sobre estas preguntas antes de continuar:

1. **¿Por qué la búsqueda binaria tiene complejidad O(log n)?**  
   *Pista: ¿cuántos elementos descarta en cada llamada?*

2. **¿Por qué `disk_usage` necesita recorrer en postorden (hijos primero, luego raíz)?**  
   *Pista: ¿para calcular el tamaño del padre necesitas saber los tamaños de los hijos?*

3. **En Hanói con n=4, ¿cuántos movimientos hay? ¿Y con n=10?**  
   *Usa la fórmula 2^n − 1 para verificar mentalmente.*

4. **¿En qué se diferencia Fibonacci de los otros tres ejemplos?**  
   *Clave: subproblemas solapados vs. subproblemas disjuntos.*

5. **Sin usar `@lru_cache`, ¿cómo implementarías memoización para Fibonacci?**  
   *Pista: un diccionario `memo = {0: 0, 1: 1}` pasado como argumento.*

```python
# Implementa aquí fib_memo_manual(n, memo=None) sin usar @lru_cache
def fib_memo_manual(n: int, memo: dict = None) -> int:
    if memo is None:
        memo = {0: 0, 1: 1}
    # Tu código aquí
    ...

# Prueba:
# assert [fib_memo_manual(i) for i in range(10)] == [0,1,1,2,3,5,8,13,21,34]
```


---

<div align="center">

**Capítulo 4 — Recursión | Notebook 2 de 3**  
*Data Structures & Algorithms in Python — Goodrich, Tamassia & Goldwasser*  
Facultad · Estructuras de Datos

[← Notebook 1: Fundamentos](01_Fundamentos_Recursion_Teoria.ipynb) &nbsp;|&nbsp; [Notebook 3: Diseño y Análisis →](03_Diseno_Analisis_Recursion_Teoria.ipynb)

</div>
